# 03 — Explainability Deep Dive
SHAP, LIME, counterfactuals, and clinical report generation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src.models.explainable_model import ExplainableMedicalAI
from src.utils.data_loader import load_sample_data
from src.utils.preprocessor import Preprocessor
from src.explainability.shap_explainer import SHAPExplainer
from src.explainability.lime_explainer import LIMEExplainer
from src.explainability.feature_importance import FeatureImportanceAnalyzer
from src.explainability.counterfactuals import CounterfactualExplainer
from src.clinical.clinical_rules import ClinicalDecisionSupport
from src.clinical.terminology_mapper import TerminologyMapper
from src.visualization.reports import ClinicalReportGenerator

In [ ]:
# Load model and data
(X_train, X_test, y_train, y_test), feature_names = load_sample_data()
prep = Preprocessor(feature_names)
X_train_s = prep.fit_transform(X_train)
X_test_s  = prep.transform(X_test)

model = ExplainableMedicalAI(model_type='decision_tree')
model.train(X_train_s, y_train, feature_names)
print('Model ready.')

In [ ]:
# Pick a high-risk patient for demonstration
# age=62, sex=M, cp=asymptomatic, bp=145, chol=280, fbs=high, maxHR=110, exang=yes, oldpeak=2.5
high_risk = np.array([62, 1, 3, 145, 280, 1, 110, 1, 2.5])
high_risk_s = prep.transform(high_risk.reshape(1, -1)).flatten()

pred  = model.predict(high_risk_s)[0]
proba = model.predict_proba(high_risk_s)[0]
print(f'Prediction : {pred}  ({"Elevated Risk" if pred==1 else "Low Risk"})')
print(f'Confidence : {proba:.1%}')

In [ ]:
# ── SHAP ──────────────────────────────────────────────────────────
shap_exp = SHAPExplainer(model.model, feature_names, X_train_s)
shap_result = shap_exp.explain(high_risk_s)

print('\nTop SHAP features:')
for f in shap_result['top_features']:
    arrow = '▲' if f['direction'] == 'increases_risk' else '▼'
    print(f"  {arrow} {f['feature']:25s}  {f['shap_value']:+.4f}")

In [ ]:
# ── LIME ──────────────────────────────────────────────────────────
lime_exp = LIMEExplainer(model.model, feature_names, X_train_s)
lime_result = lime_exp.explain(high_risk_s)

print(f'LIME fidelity (R²): {lime_result["score"]:.3f}')
print(f'Narrative: {lime_result["narrative"]}')

In [ ]:
# ── Counterfactuals ───────────────────────────────────────────────
cfe = CounterfactualExplainer(
    model.model, feature_names,
    feature_ranges={
        'age': (0, 120), 'resting_bp': (60, 200),
        'cholesterol': (100, 500), 'max_hr': (40, 210),
        'oldpeak': (0, 6),
    }
)
cfs = cfe.generate(high_risk_s, target_class=0, n_counterfactuals=3)
print(cfe.to_clinical_text(cfs))

In [ ]:
# ── Clinical Validation ───────────────────────────────────────────
cds = ClinicalDecisionSupport()
feature_dict = dict(zip(feature_names, high_risk.tolist()))
validation = cds.validate(pred, feature_dict)

print(f'Guideline adherent : {validation["guideline_adherent"]}')
print(f'Flags              : {validation["total_flags"]}')
for finding in validation['findings']:
    print(f"  [{finding['level'].upper()}] {finding['message']}")

In [ ]:
# ── Full Clinical Report ──────────────────────────────────────────
reporter = ClinicalReportGenerator()
report = reporter.generate(
    patient_id='PT-DEMO-NB03',
    features=feature_dict,
    prediction=pred,
    confidence=proba,
    shap_result=shap_result,
    lime_result=lime_result,
    guideline_result=validation,
    recommendations=reporter.generate_recommendations(pred, feature_dict),
    ci_low=max(0, proba - 0.07),
    ci_high=min(1, proba + 0.07),
)
print(reporter.to_text(report))